In [2]:
import json
import openai
import requests
from tenacity import retry, wait_random_exponential, stop_after_attempt
from termcolor import colored
import asyncio
import httpx
# from memory import MemoryManager

GPT_MODEL = "gpt-4o-mini"
GPT4_MODEL = "gpt-4o-mini"

@retry(wait=wait_random_exponential(min=1, max=40), stop=stop_after_attempt(3))
def chat_completion_request(messages, functions=None, function_call=None, model=GPT4_MODEL):
    headers = {
        "Content-Type": "application/json",
        "Authorization": "Bearer " + openai.api_key,
    }
    json_data = {"model": model, "messages": messages}
    if functions is not None:
        json_data.update({"functions": functions})
    if function_call is not None:
        json_data.update({"function_call": function_call})
    try:
        response = requests.post(
            "https://api.openai.com/v1/chat/completions",
            headers=headers,
            json=json_data,
        )
        return response
    except Exception as e:
        print("Unable to generate ChatCompletion response")
        print(f"Exception: {e}")
        return e

def pretty_print_conversation(messages):
    role_to_color = {
        "system": "red",
        "user": "green",
        "assistant": "blue",
        "function": "magenta",
    }
    formatted_messages = []
    for message in messages:
        if message["role"] == "system":
            formatted_messages.append(f"system: {message['content']}\n")
        elif message["role"] == "user":
            formatted_messages.append(f"user: {message['content']}\n")
        elif message["role"] == "assistant" and message.get("function_call"):
            formatted_messages.append(f"assistant: {message['function_call']}\n")
        elif message["role"] == "assistant" and not message.get("function_call"):
            formatted_messages.append(f"assistant: {message['content']}\n")
        elif message["role"] == "function":
            formatted_messages.append(f"function ({message['name']}): {message['content']}\n")
    for formatted_message in formatted_messages:
        print(
            colored(
                formatted_message,
                role_to_color[messages[formatted_messages.index(formatted_message)]["role"]],
            )
        )

# #leaving the async function here to make it easier for building large tests
# async def test_getFunctions(agent_response):
#     if agent_response is None:
#         return
#     if agent_response.json()["choices"][0]["message"]["function_call"] is None:
#         return
#     function_call = agent_response.json()["choices"][0]["message"]["function_call"]
#     print(function_call['arguments'])
#     data = {
#         'categories': ".",
#         'actions': q,
#         'num_results': 2,
#         'similarity_threshold': 0.7 
#     }
#     async with httpx.AsyncClient() as client:  # using httpx AsyncClient
#         response = await client.post("http://127.0.0.1:8000/get_functions/", data=data)  # async post request
#     print(response.status_code)

#     with open('tests/response_agent.json', 'a') as f:
#         f.write('\n')
#         json.dump(response.json(), f)

#     asyncio.run(test_getFunctions())

functions = [
{'name': 'query_plan_tool',
 'description': '        This is a query plan tool that takes in a list of tools and executes a query plan over these tools to answer a query. The query plan is a DAG of query nodes.\n\nGiven a list of tool names and the query plan schema, you can choose to generate a query plan to answer a question.\n\nThe tool names and descriptions are as follows:\n\n\n\n        Tool Name: aida_functions\nTool Description: Provides functions for a specified query \n        ',
 'parameters': {'title': 'QueryPlan',
  'description': "Query plan.\n\nContains a list of QueryNode objects (which is a recursive object).\nOut of the list of QueryNode objects, one of them must be the root node.\nThe root node is the one that isn't a dependency of any other node.",
  'type': 'object',
  'properties': {'nodes': {'title': 'Nodes',
    'description': 'The original question we are asking.',
    'type': 'array',
    'items': {'$ref': '#/definitions/QueryNode'}}},
  'required': ['nodes'],
  'definitions': {'QueryNode': {'title': 'QueryNode',
    'description': 'Query node.\n\nA query node represents a query (query_str) that must be answered.\nIt can either be answered by a tool (tool_name), or by a list of child nodes\n(child_nodes).\nThe tool_name and child_nodes fields are mutually exclusive.',
    'type': 'object',
    'properties': {'id': {'title': 'Id',
      'description': 'ID of the query node.',
      'type': 'integer'},
     'query_str': {'title': 'Query Str',
      'description': 'Question we are asking. This is the query string that will be executed. ',
      'type': 'string'},
     'tool_name': {'title': 'Tool Name',
      'description': 'Name of the tool to execute the `query_str`.',
      'type': 'string'},
     'dependencies': {'title': 'Dependencies',
      'description': 'List of sub-questions that need to be answered in order to answer the question given by `query_str`.Should be blank if there are no sub-questions to be specified, in which case `tool_name` is specified.',
      'type': 'array',
      'items': {'type': 'integer'}}},
    'required': ['id', 'query_str']}}}},
        {
            "name": "getInformationFromMemory",
            "description": "Retrieve historical conversational context through a semantic search. Number of tokens formula: num_chunks x (num_neighbour_chunks+1) x 256 (chunk size). Example: {\"name\": \"getInformationFromMemory\", \"parameters\": {\"conversationID\": \"123456\", \"query\": \"What was discussed earlier?\", \"num_chunks\": 5, \"num_neighbour_chunks\": 2, \"similarity_threshold\": 0.67}}",
            "parameters": {
                "type": "object",
                "properties": {
                    "conversationID": {
                        "type": "string"
                    },
                    "query": {
                        "type": "string"
                    },
                    "num_chunks": {
                        "description": "How many chunks are considered semantically. More chunks for bigger context.",
                        "type": "integer"
                    },
                    "num_neighbour_chunks": {
                        "description": "Add neighbour chunk to every semantically considered chunk. Higher value for general queries or wider range of answers.",
                        "type": "integer"
                    },
                    "similarity_threshold": {
                        "description": "Threshold for resulting chunks. Higher threshold for fewer but more precise answers.",
                        "type": "number"
                    }
                },
                "required": [
                    "conversationID",
                    "query",
                    "similarity_threshold"
                ]
            }
        },
        {
            "name": "getInformationFromUser",
            "description": "Ask for more information. Example: {\"name\": \"getInformationFromUser\", \"parameters\": {\"headline\": \"About Your Pets\", \"questions\": [{\"question\": \"What's your pet's name?\", \"input\": {\"placeholder\": \"Pet's name\"}}, {\"question\": \"What type of pet do you have?\", \"radio\": {\"labels\": [\"Dog\", \"Cat\", \"Bird\", \"Other\"]}}, {\"question\": \"When did you get your pet?\", \"date\": {}}, {\"question\": \"Does your pet have any special needs?\", \"checkbox\": {\"labels\": [\"Special Diet\", \"Medication\", \"Accessibility Needs\"]}}]}}",
            "parameters": {
                "type": "object",
                "properties": {
                    "headline": {
                        "type": "string"
                    },
                    "questions": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "question": {
                                    "type": "string"
                                },
                                "input": {
                                    "type": "object",
                                    "properties": {
                                        "placeholder": {
                                            "type": "string"
                                        }
                                    }
                                }
                            },
                            "radio": {
                                "type": "object",
                                "properties": {
                                    "labels": {
                                        "type": "array",
                                        "items": {
                                            "type": "string"
                                        }
                                    }
                                }
                            },
                            "date": {
                                "type": "object"
                            },
                            "checkbox": {
                                "type": "object",
                                "properties": {
                                    "labels": {
                                        "type": "array",
                                        "items": {
                                            "type": "string"
                                        }
                                    }
                                }
                            },
                            "required": [
                                "question"
                            ]
                        }
                    }
                },
                "required": [
                    "headline",
                    "questions"
                ]
            }
        },
        {
            "name": "getExternalFunctions",
            "description": "Obtains the external functions AiDA can execute using a semantic lookup from a list of action descriptions. If no function is returned, there's no available action for the described tasks.",
            "parameters": {
                "type": "object",
                "properties": {
                    "actions": {
                        "type": "string",
                        "description": "Comma-separated list of generalized actions AiDA might perform. The order of actions should match the order of categories. Example: 'web search,semantic analysis, retrieve financial data'."
                    },
                    "num_results_per_action": {
                        "type": "integer",
                        "default": 1,
                        "description": "Number of desired results per action. Defaults to 1, indicating a precise match."
                    },
                    "similarity_threshold": {
                        "type": "number",
                        "description": "Threshold for functions. Higher value implies more precise results."
                    },
                    "categories": {
                        "type": "string",
                        "enum": [
                            "Information Retrieval",
                            "Communication",
                            "Data Processing",
                            "Sensory Perception",
                            "Memory",
                            "Decision Making",
                            "Learning"
                        ],
                        "description": "Comma-separated list of function categories in sync with the actions order. E.g., 'Information Retrieval,Data Processing'."
                    }
                },
                "required": [
                    "actions",
                    "similarity_threshold",
                    "categories"
                ]
            }
        },
        {
            "name": "getUserIDsByCriteria",
            "description": "Get user IDs based on criteria of given information such as usernames, first names, last names, chat ids, or emails. Each entry in the array can have its own 'global', 'similarity_threshold', and 'num_results' parameters. A specific 'chat_id' can also be provided to filter users from a specific group chat. Example: {'name': 'getUserIDsByCriteria', 'parameters': {'userquery': [{'userQuery': 'john_doe', 'global': true, 'similarity_threshold': 0.71, 'num_results': 3, 'chat_id': '12345'}, {'userquery': 'jane_doe', 'global': false, 'similarity_threshold': 0.73, 'num_results': 2}]}}",
            "parameters": {
                "type": "object",
                "properties": {
                    "userQueries": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "userquery": {
                                    "type": "string"
                                },
                                "global": {
                                    "type": "boolean"
                                },
                                "similarity_threshold": {
                                    "type": "number"
                                },
                                "num_results": {
                                    "type": "integer"
                                },
                                "chat_id": {
                                    "type": "string"
                                }
                            },
                            "required": [
                                "userquery"
                            ]
                        }
                    }
                },
                "required": [
                    "userQueries"
                ]
            }
        },
        {
            "name": "summarizeText",
            "description": "Creates a summary of the provided text. The summary will be concise and highlight the key points in the text. Can be optionally used to replace a function response at a specific index with a summary to save tokens, if AiDA sees the opportunity to condense information without losing key details.",
            "parameters": {
                "type": "object",
                "properties": {
                    "text": {
                        "type": "string",
                        "description": "The text to summarize."
                    },
                    "context": {
                        "type": "string",
                        "description": "Optional. Additional context to guide the summarization."
                    },
                    "function_response_index": {
                        "type": "integer",
                        "description": "Optional. The index of the function response to replace with the summary."
                    }
                },
                "required": [
                    "text"
                ]
            }
        }
  ]

user_input = input("Write your query: ")
messages = []
messages.append({"role": "system", "content": "You're AiDA, an advanced AI assistant designed for non-technical users. Your goal is to perform actions by executing functions, not explaining them. You may use the Query plan tool to acquire external information when needed"})
messages.append({"role": "user", "content": user_input})
chat_response = chat_completion_request(
    messages, functions=functions
)
assistant_message = chat_response.json()["choices"][0]["message"]
messages.append(assistant_message)


print('------------INITIAL RESPONSE------------')
print(assistant_message)
print(chat_response)
if 'function_call' in assistant_message:
    function_call = chat_response.json()["choices"][0]["message"]["function_call"]
    print(function_call['arguments'])

------------INITIAL RESPONSE------------
{'role': 'assistant', 'content': None, 'function_call': {'name': 'getExternalFunctions', 'arguments': '{\n  "actions": "send crypto",\n  "similarity_threshold": 0.9,\n  "categories": "Communication"\n}'}}
<Response [200]>
{
  "actions": "send crypto",
  "similarity_threshold": 0.9,
  "categories": "Communication"
}


# Query Planner

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from llama_index import (
    SimpleDirectoryReader,
    LLMPredictor,
    ServiceContext,
    GPTVectorStoreIndex,
)
from llama_index.response.pprint_utils import pprint_response
from llama_index.llms import OpenAI

/opt/homebrew/lib/python3.11/site-packages/deeplake/util/check_latest_version.py:32: UserWarning: A newer version of deeplake (3.6.12) is available. It's recommended that you update to the latest version using `pip install -U deeplake`.
  warnings.warn(


In [3]:
from langchain.vectorstores import Pinecone
from langchain_openai import OpenAIEmbeddings

In [4]:
llm = OpenAI(temperature=0, model="gpt-4o-mini")
service_context = ServiceContext.from_defaults(llm=llm)

In [5]:
functions_db = SimpleDirectoryReader(
    input_files=["functions.json"]
).load_data()

In [6]:
functions_index = GPTVectorStoreIndex.from_documents(functions_db)

In [7]:
pinecone_functions = functions_index.as_query_engine(
    similarity_top_k=3, service_context=service_context
)

In [8]:
from llama_index.tools import QueryEngineTool


query_tool = QueryEngineTool.from_defaults(
    query_engine=pinecone_functions,
    name="aida_functions",
    description=f"Provides functions for a specified query",
)

In [9]:
# define query plan tool
from llama_index.tools import QueryPlanTool
from llama_index import get_response_synthesizer

response_synthesizer = get_response_synthesizer(service_context=service_context)
query_plan_tool = QueryPlanTool.from_defaults(
    query_engine_tools=[query_tool],
    response_synthesizer=response_synthesizer,
)

In [10]:
query_plan_tool.metadata.to_openai_function()

{'name': 'query_plan_tool',
 'description': '        This is a query plan tool that takes in a list of tools and executes a query plan over these tools to answer a query. The query plan is a DAG of query nodes.\n\nGiven a list of tool names and the query plan schema, you can choose to generate a query plan to answer a question.\n\nThe tool names and descriptions are as follows:\n\n\n\n        Tool Name: aida_functions\nTool Description: Provides functions for a specified query \n        ',
 'parameters': {'title': 'QueryPlan',
  'description': "Query plan.\n\nContains a list of QueryNode objects (which is a recursive object).\nOut of the list of QueryNode objects, one of them must be the root node.\nThe root node is the one that isn't a dependency of any other node.",
  'type': 'object',
  'properties': {'nodes': {'title': 'Nodes',
    'description': 'The original question we are asking.',
    'type': 'array',
    'items': {'$ref': '#/definitions/QueryNode'}}},
  'required': ['nodes'],

In [11]:
from llama_index.agent import OpenAIAgent
from llama_index.llms import OpenAI


agent = OpenAIAgent.from_tools(
    [query_plan_tool],
    max_function_calls=10,
    llm=OpenAI(temperature=0, model="gpt-4o-mini"),
    verbose=True,
)

In [14]:
response = agent.query("can you get the weather in london?")

In [15]:
response

Response(response="Sure, I can help with that. However, as a text-based AI model, I don't have real-time capabilities to fetch live data such as current weather conditions. I recommend using a weather forecasting website or app like the Weather Channel, BBC Weather, or a similar service for the most accurate and up-to-date information.", source_nodes=[], metadata=None)